In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

BASE_DIR = Path(r"output\data\step_3.2")

TXT_DIR = Path(r"output\reports\3.4.2_txt")
TXT_DIR.mkdir(parents=True, exist_ok=True)

CSV_DIR = Path(r"output\reports\3.4.2_csv")
CSV_DIR.mkdir(parents=True, exist_ok=True)

HTML_DIR = Path(r"output\reports\3.4.2_html")
HTML_DIR.mkdir(parents=True, exist_ok=True)

print("Cartella corrente:", BASE_DIR.resolve())

print("Cartella TXT:", TXT_DIR.resolve())
print("Cartella CSV:", CSV_DIR.resolve())
print("Cartella HTML:", HTML_DIR.resolve())



In [15]:
from pathlib import Path
import pandas as pd

def must_load_csv(base_dir, filename):
    path = Path(base_dir) / filename
    if not path.exists():
        raise FileNotFoundError(f'File non trovato: {path}')
    print(f'Caricato: {path}')
    return pd.read_csv(path)

prospetto_ambito_per_regione = must_load_csv(BASE_DIR, "prospetto_ambito_per_regione.csv")
prospetto_attori_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_attori_tipo10_per_regione.csv")
prospetto_firmatari_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_firmatari_tipo10_per_regione.csv")
prospetto_governance_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_governance_tipo10_per_regione.csv")
prospetto_proponenti_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_proponenti_tipo10_per_regione.csv")
prospetto_reti_per_regione = must_load_csv(BASE_DIR, "prospetto_reti_per_regione.csv")
prospetto_soggetti_tipo10_per_regione = must_load_csv(BASE_DIR, "prospetto_soggetti_tipo10_per_regione.csv")
prospetto_soggetti_tipo30_per_regione = must_load_csv(BASE_DIR, "prospetto_soggetti_tipo30_per_regione.csv")
tabella_reti = must_load_csv(BASE_DIR, "tabella_reti.csv")
tabella_soggetti = must_load_csv(BASE_DIR, "tabella_soggetti.csv")

loaded_tables = {
    "Ambito per Regione": prospetto_ambito_per_regione,
    "Attori per Regione × aggregazione 10": prospetto_attori_tipo10_per_regione,
    "Firmatari per Regione × aggregazione 10": prospetto_firmatari_tipo10_per_regione,
    "Governance per Regione × aggregazione 10": prospetto_governance_tipo10_per_regione,
    "Proponenti per Regione × aggregazione 10": prospetto_proponenti_tipo10_per_regione,
    "Reti per Regione": prospetto_reti_per_regione,
    "Soggetti per Regione × aggregazione 10": prospetto_soggetti_tipo10_per_regione,
    "Soggetti per Regione × Classe 30": prospetto_soggetti_tipo30_per_regione,
    "Tabella Reti": tabella_reti,
    "Tabella Soggetti": tabella_soggetti,
}

print('\nTabelle caricate:', len(loaded_tables))

Caricato: output\data\step_3.2\prospetto_ambito_per_regione.csv
Caricato: output\data\step_3.2\prospetto_attori_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_firmatari_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_governance_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_proponenti_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_reti_per_regione.csv
Caricato: output\data\step_3.2\prospetto_soggetti_tipo10_per_regione.csv
Caricato: output\data\step_3.2\prospetto_soggetti_tipo30_per_regione.csv
Caricato: output\data\step_3.2\tabella_reti.csv
Caricato: output\data\step_3.2\tabella_soggetti.csv

Tabelle caricate: 10


In [ ]:
def add_totals(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    out = df.copy()

    if "regione" not in out.columns:
        raise ValueError("La tabella non contiene la colonna 'regione'")

    num_cols = [c for c in out.columns if c != "regione"]

    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    out["Totale Riga"] = out[num_cols].sum(axis=1)

    total_row = {"regione": "TOTALE COLONNA"}
    for c in num_cols:
        total_row[c] = out[c].sum()
    total_row["Totale Riga"] = out["Totale Riga"].sum()

    out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)
    return out


def style_table(df: pd.DataFrame):
    display(
        df.style
        .format(precision=0)
        .set_properties(**{"text-align": "center"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]},
        ])
    )


In [ ]:
#tbl = add_totals(prospetto_soggetti_tipo10_per_regione)
#style_table(tbl)
for table in loaded_tables.values():
    tbl = add_totals(table)
    title = [k for k, v in loaded_tables.items() if v is table][0]
    print(f"\n=== {title} ===")
    style_table(tbl)

In [ ]:
from pathlib import Path
import pandas as pd

def export_table_csv_with_totals(
    df: pd.DataFrame,
    output_csv,
    row_label_col=None,
    index=False
) -> pd.DataFrame:
    """
    Salva una tavola con:
    - Totale Riga
    - Totale Colonna
    - Totale Generale

    Parametri
    ---------
    df : DataFrame
    output_csv : path file csv
    row_label_col : nome colonna etichette righe (opzionale)
    """

    if df is None or df.empty:
        raise ValueError("DataFrame vuoto")

    out = df.copy()

    # se non indicata: usa prima colonna
    if row_label_col is None:
        row_label_col = out.columns[0]

    if row_label_col not in out.columns:
        raise ValueError(f"Colonna '{row_label_col}' non trovata")

    # colonne numeriche = tutte tranne etichetta
    num_cols = [c for c in out.columns if c != row_label_col]

    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    # totale riga
    out["Totale Riga"] = out[num_cols].sum(axis=1)

    # riga totale colonna
    total_row = {row_label_col: "TOTALE COLONNA"}

    for c in num_cols:
        total_row[c] = out[c].sum()

    total_row["Totale Riga"] = out["Totale Riga"].sum()

    out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)

    # salva
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=index, encoding="utf-8-sig")

    print(f"Creato: {output_csv}")
    return out

# html & txt table

In [ ]:
from pathlib import Path
import pandas as pd
from html import escape



files = [
    ("prospetto_ambito_per_regione.csv", "Ambito per Regione"),
    
    ("prospetto_attori_tipo10_per_regione.csv", "Attori per Regione × Aggregazione 10"),
    ("prospetto_firmatari_tipo10_per_regione.csv", "Firmatari per Regione × Aggregazione 10"),
    ("prospetto_proponenti_tipo10_per_regione.csv", "Proponenti per Regione × Aggregazione 10"),
     
    ("prospetto_governance_tipo10_per_regione.csv", "Governance per Regione × Aggregazione 10"),
  
    ("prospetto_reti_per_regione.csv", "Reti per Regione"),    
    ("tabella_reti.csv", "Tabella Reti"),

    ("prospetto_soggetti_tipo10_per_regione.csv", "Soggetti per Regione × Aggregazione 10"),
    ("prospetto_soggetti_tipo30_per_regione.csv", "Soggetti per Regione × Classe 30"),
    ("tabella_soggetti.csv", "Tabella Soggetti"),
]

# =========================================================
# CONFIGURAZIONE
# =========================================================

INCLUDE_TABLES = [
    # lascia vuoto [] per includere tutto
]

EXCLUDE_TABLES = [
    # esempio:
    # "Gestione per Regione × Aggregazione 10",
    # "Coordinamento per Regione × Aggregazione 10",
]

TABLE_COLUMN_RULES = {
    # esempio:
    # "Attori per Regione × Aggregazione 10": {
    #     "keep_columns": ["regione", "CAV", "Servizi comunali", "associazionismo", "Totale Riga"],
    #     "drop_columns": [],
    #     "sort_by": "Totale Riga",
    #     "ascending": False,
    #     "drop_zero_columns": True,
    # },

    # "Soggetti per Regione × Classe 30": {
    #     "keep_columns": None,
    #     "drop_columns": ["NON CLASSIFICATO"],
    #     "sort_by": "Totale Riga",
    #     "ascending": False,
    #     "drop_zero_columns": True,
    # },
}


def table_is_selected(title: str) -> bool:
    if INCLUDE_TABLES and title not in INCLUDE_TABLES:
        return False
    if title in EXCLUDE_TABLES:
        return False
    return True

def add_totals(df):
    out = df.copy()

    key_col = out.columns[0]
    num_cols = [c for c in out.columns if c != key_col]

    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    # SOMMA ORIZZONTALE = Totale Riga
    out["Totale Riga"] = out[num_cols].sum(axis=1)

    # SOMMA VERTICALE = Totale Colonna
    total_row = {key_col: "Totale Colonna"}
    for c in num_cols:
        total_row[c] = out[c].sum()

    total_row["Totale Riga"] = out["Totale Riga"].sum()

    out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)
    return out

def _add_totals(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "regione" not in df.columns:
        return df.copy()

    out = df.copy()
    num_cols = [c for c in out.columns if c != "regione"]

    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    numeric_subset = [c for c in num_cols if pd.api.types.is_numeric_dtype(out[c])]

    out["Totale Riga"] = out[numeric_subset].fillna(0).sum(axis=1)

    total_row = {"regione": "TOTALE COLONNA"}
    for c in out.columns:
        if c == "regione":
            continue
        if c in numeric_subset or c == "Totale Riga":
            total_row[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).sum()
        else:
            total_row[c] = ""

    return pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)


def apply_table_rules(df: pd.DataFrame, title: str) -> pd.DataFrame:
    out = df.copy()
    rules = TABLE_COLUMN_RULES.get(title, {})

    keep_columns = rules.get("keep_columns")
    drop_columns = rules.get("drop_columns", [])
    sort_by = rules.get("sort_by")
    ascending = rules.get("ascending", False)
    drop_zero_columns = rules.get("drop_zero_columns", False)

    if keep_columns:
        keep_columns = [c for c in keep_columns if c in out.columns]
        out = out[keep_columns]

    if drop_columns:
        drop_columns = [c for c in drop_columns if c in out.columns]
        out = out.drop(columns=drop_columns)

    if drop_zero_columns:
        cols_to_keep = []
        for c in out.columns:
            if c == "regione":
                cols_to_keep.append(c)
                continue

            col_num = pd.to_numeric(out[c], errors="coerce")
            if col_num.fillna(0).sum() != 0:
                cols_to_keep.append(c)

        out = out[cols_to_keep]

    if sort_by and sort_by in out.columns:
        if "regione" in out.columns:
            regione_str = out["regione"].astype(str).str.strip()

            mask_total = regione_str.eq("TOTALE COLONNA")
            df_regioni = out.loc[~mask_total].copy()
            df_totale = out.loc[mask_total].copy()

            df_regioni[sort_by] = pd.to_numeric(df_regioni[sort_by], errors="coerce").fillna(0)
            df_regioni = df_regioni.sort_values(by=sort_by, ascending=ascending)

            out = pd.concat([df_regioni, df_totale], ignore_index=True)
        else:
            out[sort_by] = pd.to_numeric(out[sort_by], errors="coerce").fillna(0)
            out = out.sort_values(by=sort_by, ascending=ascending)

    return out


loaded = []
for fname, title in files:
    path = BASE_DIR / fname
    if path.exists():
        df = pd.read_csv(path)
        loaded.append((fname, title, df))

# TXT
txt_parts = []
txt_parts.append("REPORT TABELLARE CSV - RETI VIOLENZA\n")
for fname, title, df in loaded:
    if not table_is_selected(title):
        continue

    txt_parts.append("=" * 100)
    txt_parts.append(title)
    txt_parts.append(f"FILE: {fname}")
    txt_parts.append(f"RIGHE: {len(df)} | COLONNE: {len(df.columns)}")
    txt_parts.append("")

    table_df = add_totals(df)
    table_df = apply_table_rules(table_df, title)

    with pd.option_context("display.max_columns", None, "display.width", 240):
        txt_parts.append(table_df.to_string(index=False))
    txt_parts.append("")



txt_path = TXT_DIR / "report_tabellare_csv_stampabile.txt"

txt_path.write_text("\n".join(txt_parts), encoding="utf-8")

# HTML
css = """
body { font-family: Arial, sans-serif; margin: 24px; color: #111; }
h1 { margin-bottom: 8px; }
h2 { margin-top: 36px; margin-bottom: 8px; page-break-before: always; }
.meta { color: #444; margin-bottom: 10px; }
table { border-collapse: collapse; width: 100%; font-size: 12px; }
th, td { border: 1px solid #999; padding: 6px 8px; text-align: center; }
th { background: #efefef; position: sticky; top: 0; }
tr:last-child td { font-weight: bold; background: #f7f7f7; }
.section { margin-bottom: 28px; }
.small { font-size: 11px; color: #555; }
@media print {
  body { margin: 10mm; }
  h2 { page-break-before: always; }
  table { font-size: 10px; }
}
"""
html_parts = [
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    f"<style>{css}</style></head><body>",
    "<h1>Report tabellare CSV – Reti violenza</h1>",
    "<div class='meta'>Tabelle generate dai CSV caricati. Include totali di riga, totali di colonna e totale generale quando presente la colonna <b>regione</b>.</div>"
]


csv_out = Path("output/reports/3.4.2_csv/")
csv_out.mkdir(parents=True, exist_ok=True)

for fname, title, df in loaded:
    if not table_is_selected(title):
        continue

    table_df = add_totals(df)
    table_df = apply_table_rules(table_df, title)

    # export cvs con totali
    table_df.to_csv(
        csv_out / f"{Path(fname).stem}_totali.csv",
        index=False,
        encoding="utf-8-sig"
    )


    html_parts.append("<div class='section'>")
    html_parts.append(f"<h2>{escape(title)}</h2>")
    html_parts.append(f"<div class='small'>File: {escape(fname)} | Righe: {len(df)} | Colonne: {len(df.columns)}</div>")
    html_parts.append(table_df.to_html(index=False, border=0, na_rep="", escape=True))
    html_parts.append("</div>")

html_parts.append("</body></html>")
html_path = HTML_DIR/ "report_tabellare_csv_stampabile.html"
html_path.parent.mkdir(parents=True, exist_ok=True)
html_path.write_text("\n".join(html_parts), encoding="utf-8")

print(txt_path)
print(html_path)